> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 12 · Notebook 04 — Option position management and the trade journal

**Sessions:** S7–S8 (Option strategy integration · Trade journal & journal analytics) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Manage a short-premium option position with prioritized rules: stop, take profit, expiry, ex-dividend, delta, roll.
2. Measure every trade in R-multiples.
3. Find the edge that only shows up by setup and regime.
4. Catch rule violations in the journal.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Rules, in priority order

A short-premium position (credit received, cost to close now = `value`) is checked every bar. The **order** matters: a position that is both at its stop and near ex-dividend must be closed as a stop. After stop, take profit and expiry (given): **ex-dividend** (a short call is in the money and ex-dividend is at most 1 day away: early-assignment risk → close), then **adjust** (any |short delta| above `self.max_delta`), then **roll** (days to expiry ≤ `self.roll_dte`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyOptionPositionManager(p.OptionPositionManager):
    def evaluate(self, pos):
        if pos["value"] >= pos["credit"] * (1 + self.stop):
            return "close", "stop"
        if pos["value"] <= pos["credit"] * (1 - self.tp):
            return "close", "take_profit"
        if pos["dte"] == 0:
            return "close", "expiry"
        dx = pos.get("days_to_exdiv")
        if pos.get("short_call_itm") and dx is not None and dx <= 1:
            return "close", "ex_dividend"
        if any(abs(d) > self.max_delta for d in pos["short_deltas"]):
            return "adjust", "short_delta"
        if pos["dte"] <= self.roll_dte:
            return "roll", "dte"
        return None

base = {"credit": 2.0, "value": 1.6, "dte": 30, "short_deltas": [0.20, -0.18], "short_call_itm": False, "days_to_exdiv": None}
scenarios = {"nothing to do": base, "loss of 2× the credit": {**base, "value": 6.1}, "half the credit captured": {**base, "value": 0.9},
             "short delta 0.35": {**base, "short_deltas": [0.35, -0.10]}, "21 days left": {**base, "dte": 21},
             "short call ITM, ex-div tomorrow": {**base, "short_call_itm": True, "days_to_exdiv": 1, "short_deltas": [0.6, -0.1]},
             "ITM call, ex-div tomorrow AND big loss": {**base, "value": 6.5, "short_call_itm": True, "days_to_exdiv": 1}}
mine = {k: p.attempt(MyOptionPositionManager().evaluate, v) for k, v in scenarios.items()}
mine = p.check("OptionPositionManager.evaluate", mine, {k: p.OptionPositionManager().evaluate(v) for k, v in scenarios.items()})
pd.Series({k: v if v else "—" for k, v in mine.items()}, name="(action, reason)").to_frame()

## 2. R-multiples

Measure every trade in units of the risk you **planned**: `R = side·(exit − entry) / |entry − stop|`. A trade without a stop has no defined risk, so its R is NaN (and it is a rule violation).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def r_multiples(journal):
    return journal["side"] * (journal["exit"] - journal["entry"]) / (journal["entry"] - journal["stop"]).abs()

journal = p.trade_journal()                                          # 400 trades with a PLANTED edge
mine = p.check("r_multiples", p.attempt(r_multiples, journal), p.r_multiples(journal))
journal = journal.assign(r_multiple=mine)
print(f"{int(mine.isna().sum())} trades have no stop, so no R;  mean R of the rest: {mine.mean():+.3f}")

## 3. Where is the edge?

`p.journal_stats` gives trades, win rate, average win and loss in R, and **expectancy** (mean R) per group. By setup alone, then by setup and regime:

In [ ]:
display(p.journal_stats(journal).round(3))
p.journal_stats(journal, ["setup_tag", "regime"]).round(3)

By setup the two look alike. Split by regime, breakouts earn a lot in trends and lose in chop, while pullbacks earn a little everywhere. The action: take breakouts only in the trend regime (a filter the regime model of Part 10 can supply), and journal the regime at entry so the split is possible.

## 4. Rule violations

The journal is also a compliance record: trades without a stop, entries outside market hours, positions larger than planned (`p.rule_violations`):

In [ ]:
violations = p.rule_violations(journal)
display(violations["rule"].value_counts().to_frame("trades"))
violations.head()

## Wrap-up

* Option management rules are prioritized; test combinations, not single conditions.
* R-multiples make trades comparable; no stop, no R.
* Edges hide in segments: journal the setup and the regime.
* Graded version: `labs/part12/week42_execution` (`OptionPositionManager`, `r_multiples`, `journal_stats`, `rule_violations`).